# Surface Plotting Framework

Hands-on tour of the new `mindmap.viz` 3D framework. It replaces the old `plot_brain_activity`/`plot_source_estimate`/etc. (kept for reference in `mindmap/viz/OLD/`) with three tiers:

- `Layer` (one mesh + an optional scalar field) and `Scene` (a multi-panel interactive plotter) - the low-level primitives, maximum flexibility.
- `Panel` typed shortcuts (e.g. `SurfacePlot`) - clean, named parameters for *one* plot, each fully independent (its own `mesh_params`/`func_params` - no more parallel lists of dicts to keep in sync).
- Combine panels with operators instead of index-aligned lists: `+` merges panels into the **same** subplot, `|` lays panels **side by side**, `&` **stacks** them vertically (data model ready, rendering not implemented yet).

Covers the four functionalities the framework was built for:

1. a surface with a static map
2. multiple surfaces side by side (`|`)
3. a surface with a temporal (or other) axis map
4. different surfaces/point clouds combined in one panel (`+`)

Synthetic spheres first (fast, no data needed), then the same primitives applied to a real subject - including plotting a **single cortical hemisphere on its own**, which the old framework couldn't do (an `mne.SourceSpaces` always bundles both hemispheres together).

## 🧪 - Setup

In [16]:
import numpy as np
import pyvista as pv

import mindmap as mm
from mindmap.viz import SurfacePlot, BrainPlot, Layer, Scene
from mindmap.geometry.io import surf_to_polydata
from mindmap.anatomy.source_space.process import build_dense_vertex_map
from mindmap.core.process import paint_on_dense_mesh
from mindmap.analysis.functional.eeg.io import get_ch_positions

dm = mm.io.TutoManager("../config.yml")


## 📊 - Synthetic surfaces: the four core functionalities

### 1. A surface with a static map

In [17]:
sphere = pv.Icosphere(radius=1.0, nsub=3)
values = sphere.points[:, 2]  # color by height (z)

SurfacePlot(sphere, values, title="Static map").show()


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad3665b49d0_15&reconnect=auto" class="pyv…

### 2. Multiple surfaces side by side - the `|` operator

Each `SurfacePlot` is an independent object: its own mesh, values, `mesh_params`, `func_params`. `|` just lays them out - no parallel lists to keep aligned.

**A note on `scalar_name`**: it defaults to the same literal `"value"` for every `SurfacePlot` - simple and predictable, no generated names to guess when selecting with `link_scale` later. The trade-off: PyVista synchronizes the color range of every actor sharing a scalar array name within one `Plotter` (even across subplots) to whichever was added *last*. So combining several default-named plots with different data ranges would make them all silently use the last one's range - give each plot its own `scalar_name` whenever you're comparing more than one, like below.

In [18]:
small  = SurfacePlot(pv.Icosphere(radius=0.6, nsub=3), title="small", scalar_name="small")
medium = SurfacePlot(pv.Icosphere(radius=1.0, nsub=3), title="medium", scalar_name="medium")
large  = SurfacePlot(pv.Icosphere(radius=1.4, nsub=3), title="large", scalar_name="large")

for plot in (small, medium, large):
    plot.values = plot.mesh.points[:, 0]  # color by x

(small | medium | large).show()


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad36692f5d0_16&reconnect=auto" class="pyv…

### `&` - vertical stacking

`&` adds a new row instead of extending the current one. Every row must end up with the same number of panels (a rectangular grid) - `|`/`&` always produce that naturally when chained normally.

In [19]:
(small & medium & large).show()  # a 3-row, 1-column stack


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad3665fa0d0_17&reconnect=auto" class="pyv…

Rows and columns combine freely - `(a | b) & (c | d)` is a genuine 2x2 grid, one shared slider still steps every panel.

In [20]:
top_left  = SurfacePlot(pv.Icosphere(radius=1.0, nsub=2), title="top-left")
top_right = SurfacePlot(pv.Icosphere(radius=1.0, nsub=2), title="top-right")
bot_left  = SurfacePlot(pv.Icosphere(radius=1.0, nsub=2), title="bottom-left")
bot_right = SurfacePlot(pv.Icosphere(radius=1.0, nsub=2), title="bottom-right")

for plot in (top_left, top_right, bot_left, bot_right):
    plot.values = plot.mesh.points[:, 2]  # color by z

((top_left | top_right) & (bot_left | bot_right)).show()


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad365a4fe50_18&reconnect=auto" class="pyv…

### Different parameters per panel

This is the case that used to require `mesh_params=[{...}, {...}]`/`func_params=[{...}, {...}]` index-aligned lists. Now each panel just owns its own dict as a normal constructor argument.

In [21]:
sphere_a, sphere_b = pv.Icosphere(radius=1.0, nsub=3), pv.Icosphere(radius=1.0, nsub=3)

raw    = SurfacePlot(sphere_a, np.random.randn(sphere_a.n_points), title="raw",
                      mesh_params={"opacity": 1.0}, scalar_name="raw")
abs_translucent = SurfacePlot(sphere_b, np.random.randn(sphere_b.n_points), title="abs + translucent",
                               func_params={"absolute": True}, mesh_params={"opacity": 0.5}, scalar_name="abs")

(raw | abs_translucent).show()


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad366ae4150_19&reconnect=auto" class="pyv…

### `func_params` - preprocessing before plotting

`normalize` (`"max"`/`"minmax"`/`"zscore"`) and `absolute` are applied to `values` before the mesh is painted, per panel - handy for comparing quantities that live on different scales without touching the underlying data.

`a`/`b` get distinct `scalar_name`s below so each renders on its *own* correct range (they're deliberately not comparable yet). `a_norm`/`b_norm` are given the *same* name instead - once normalized they genuinely are comparable, so sharing PyVista's native color range is exactly what we want, with no extra step needed (that's the same "same name = deliberately linked" signal `link_scale` uses).

Both pairs reuse the same `pv.PolyData` object (`sphere_c`/`sphere_d`) rather than each getting its own copy - safe on PyVista >= 0.48 (this environment's version), which fixed [pyvista#1900](https://github.com/pyvista/pyvista/issues/1900) (a shared mesh's `point_data`/active-scalars used to get mutated when plotted more than once with different scalar arrays). On an older PyVista, give each plot its own `mesh.copy()` instead.

In [22]:
sphere_c, sphere_d = pv.Icosphere(radius=1.0, nsub=3), pv.Icosphere(radius=1.0, nsub=3)

unscaled = np.random.randn(sphere_c.n_points) * 50   # a much larger range
tiny     = np.random.randn(sphere_d.n_points) * 0.01  # a much smaller range

a = SurfacePlot(sphere_c, unscaled, title="raw (x50)", scalar_name="a")
b = SurfacePlot(sphere_d, tiny, title="raw (x0.01)", scalar_name="b")
a_norm = SurfacePlot(sphere_c, unscaled, title="zscore-normalized",
                      func_params={"normalize": "zscore"}, scalar_name="normalized")
b_norm = SurfacePlot(sphere_d, tiny, title="zscore-normalized",
                      func_params={"normalize": "zscore"}, scalar_name="normalized")

(a | b).show()             # each on its own scale - both show real structure
(a_norm | b_norm).show()   # same scalar_name -> comparable on one shared scale


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad352e8bb90_20&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad352d53490_21&reconnect=auto" class="pyv…

### 3. A surface with a temporal (or other) axis map

In [23]:
sphere = pv.Icosphere(radius=1.0, nsub=3)
times = np.linspace(0, 1, 30)

# a wave traveling around the sphere over time
theta = np.arctan2(sphere.points[:, 1], sphere.points[:, 0])
wave = np.sin(theta[:, None] - 2 * np.pi * times[None, :])

SurfacePlot(sphere, wave, states=times, title="Traveling wave").show(slider_label="Time [s]")


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad352c2ac50_22&reconnect=auto" class="pyv…

### 3b. A non-temporal state axis - e.g. a harmonic index

`states` doesn't have to be time - it's just whatever axis the slider steps through. Here each state is a different spatial frequency mode, labeled by its harmonic index rather than seconds (this is exactly the shape connectome eigenvector sweeps take).

In [24]:
sphere = pv.Icosphere(radius=1.0, nsub=3)
harmonics = np.arange(1, 6)  # mode index, not time

theta = np.arctan2(sphere.points[:, 1], sphere.points[:, 0])
modes = np.stack([np.cos(k * theta) for k in harmonics], axis=1)

SurfacePlot(sphere, modes, states=harmonics, title="Synthetic harmonics").show(slider_label="Harmonic index")


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad352cd9310_23&reconnect=auto" class="pyv…

### 4. Different surfaces (incl. point clouds) combined in one panel - the `+` operator

`+` merges panels into the *same* subplot instead of laying them out separately. Style/scalar handling (surface vs. point cloud) is auto-detected per mesh, same as before.

In [25]:
sphere = pv.Icosphere(radius=1.0, nsub=3)
surface = SurfacePlot(sphere, mesh_params={"color": "white", "opacity": 0.3}, title="surface + point cloud")

n_points = 40
cloud_pos = 1.3 * sphere.points[np.random.choice(sphere.n_points, n_points, replace=False)]
cloud = SurfacePlot(pv.PolyData(cloud_pos), np.random.randn(n_points), mesh_params={"cmap": "bkr"})

(surface + cloud).show()


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad352ba2050_24&reconnect=auto" class="pyv…

### Bonus: adding one panel to every panel of a layout

`+` between a multi-panel `Layout` and a single `Panel` broadcasts that panel into every one of them (still the same number of panels, each one now a merge) - handy for e.g. dropping the same marker/overlay onto several comparison plots at once.

In [26]:
marker_pos = np.array([[0.0, 0.0, 1.4]])  # a single point "north pole" marker
marker = SurfacePlot(pv.PolyData(marker_pos), mesh_params={"color": "black", "point_size": 20})

((small | medium | large) + marker).show()


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad352a5bc10_25&reconnect=auto" class="pyv…

### Comparable color scales - `Layout.link_scale`

`+` (same panel) almost never wants a shared scale - you merge brain + EEG precisely because they're different quantities. `|` (side by side) is where it's ambiguous: sometimes you want independent scales, sometimes (comparing the same quantity across panels) you want them locked together. `link_scale` selects by `scalar_name`, so it can link just the parts that represent the same thing and leave the rest alone - e.g. two "brain + sensor" panels where only the brain should be compared on the same scale, not the (differently-united) EEG vs. iEEG readings next to them.

In [27]:
sphere = pv.Icosphere(radius=1.0, nsub=3)

brain_a = SurfacePlot(sphere, np.random.randn(sphere.n_points) * 1, title="A", scalar_name="Source Activity")
eeg_a   = SurfacePlot(pv.PolyData(1.3 * sphere.points[:20]), np.random.randn(20), scalar_name="EEG Potential")

brain_b = SurfacePlot(sphere, np.random.randn(sphere.n_points) * 5, title="B", scalar_name="Source Activity")
ieeg_b  = SurfacePlot(pv.PolyData(1.3 * sphere.points[20:40]), np.random.randn(20), scalar_name="iEEG Potential")

layout = (brain_a + eeg_a) | (brain_b + ieeg_b)
layout.link_scale("Source Activity")  # only the brain layers get a shared range

layout.show()


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad35296b950_26&reconnect=auto" class="pyv…

### Two independently-linked groups

`link_scale` is chainable, and each call resolves its name(s) independently - link the brain layers to one range, the sensor layers (if they *do* share a unit this time) to a different one, in the same layout.

In [28]:
sphere = pv.Icosphere(radius=1.0, nsub=3)

brain_a = SurfacePlot(sphere, np.random.randn(sphere.n_points) * 1, title="A", scalar_name="Source Activity")
eeg_a   = SurfacePlot(pv.PolyData(1.3 * sphere.points[:20]), np.random.randn(20) * 1, scalar_name="EEG Potential")

brain_b = SurfacePlot(sphere, np.random.randn(sphere.n_points) * 5, title="B", scalar_name="Source Activity")
eeg_b   = SurfacePlot(pv.PolyData(1.3 * sphere.points[:20]), np.random.randn(20) * 3, scalar_name="EEG Potential")

layout = (brain_a + eeg_a) | (brain_b + eeg_b)
layout.link_scale("Source Activity").link_scale("EEG Potential")  # two separate shared ranges

layout.show()


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad35287f050_27&reconnect=auto" class="pyv…

## 🧠 - Real data: a single cortical hemisphere

The old framework's `BrainMesh`/`build_hemi_pair` always split an `mne.SourceSpaces` into *both* hemispheres before plotting - there was no way to render just one. `Layer`/`SurfacePlot` don't know about hemispheres at all: hand them whichever `pv.PolyData` you built, and that's what gets plotted.

`stc` is cropped to 100 timepoints (~2.0s +/- 0.25s) rather than its full 801 - on the Jupyter/trame backend, every slider drag re-renders and streams the updated ~291k-point brain mesh to the browser, so a shorter, more manageable time axis keeps this notebook responsive. This is a notebook-only concern, not a framework limitation - `states`/the slider itself works identically at any length.

In [29]:
src = dm.load("intra12", "src")
stc = dm.load("intra12", "stc", desc="wcset").crop(tmin=1.75, tmax=2.245)  # 100 timepoints around 2s

split = src[0]["nuse"]                       # left/right dipole split
map_l = build_dense_vertex_map(src[0])        # dense-vertex -> nearest dipole
hemi_l = surf_to_polydata(src[0]["rr"], src[0]["tris"])

mid = len(stc.times) // 2
snapshot = paint_on_dense_mesh(stc.data[:split, mid], map_l)

SurfacePlot(hemi_l, snapshot, title=f"Left hemisphere @ t={stc.times[mid]:.3f}s").show()


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad3528bd450_28&reconnect=auto" class="pyv…

### Both hemispheres, side by side, a short time course

`SurfacePlot.values` also accepts a *callable* provider instead of a plain array - here each hemisphere paints its dipole-space `stc` data onto its dense mesh lazily, one timepoint at a time, instead of precomputing every timepoint up front. This dipole-mapping logic lives entirely in this notebook cell, not inside the framework - that separation is the point of the redesign.

Both panels are named `scalar_name="Source Activity"` and linked with `.link_scale(...)` so they share one color scale - since the values come from callable providers (can't be inspected automatically), we pass the range explicitly.

In [30]:
hemi_r = surf_to_polydata(src[1]["rr"], src[1]["tris"])
map_r = build_dense_vertex_map(src[1])

def provider_l(idx):
    return paint_on_dense_mesh(stc.data[:split, idx], map_l)

def provider_r(idx):
    return paint_on_dense_mesh(stc.data[split:, idx], map_r)

lh = SurfacePlot(hemi_l, provider_l, states=stc.times, title="lh", scalar_name="Source Activity")
rh = SurfacePlot(hemi_r, provider_r, states=stc.times, title="rh", scalar_name="Source Activity")

layout = lh | rh
layout.link_scale("Source Activity", clim=(0, np.abs(stc.data).max()), cmap="Reds")
layout.show(slider_label="Time [s]")


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad3583a3fd0_29&reconnect=auto" class="pyv…

### Brain surface + electrode point cloud

Functionality 4 again (`+`), this time with real intracranial EEG electrodes over a hemisphere, both driven by the same time slider. The brain and the electrodes get different `scalar_name`s (different quantities, different units) - if we later put this panel side by side with another subject's, `link_scale("Source Activity")` would link only the brain layers, leaving each panel's own electrode scale alone.

In [31]:
ave = dm.load("intra12", "ave", task="spk").crop(tmin=1.75, tmax=2.245)  # same 100-timepoint window as stc
positions = get_ch_positions(ave)

brain = SurfacePlot(hemi_l, mesh_params={"color": "white", "opacity": 0.3}, title="lh + iEEG electrodes",
                     scalar_name="Source Activity")
electrodes = SurfacePlot(pv.PolyData(positions), ave.data, states=ave.times, mesh_params={"cmap": "bkr"},
                          scalar_name="iEEG Potential")

(brain + electrodes).show(slider_label="Time [s]")


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad366bc6b90_30&reconnect=auto" class="pyv…

## 🧠 - `BrainPlot`: the brain-specific shortcut

Everything above was built directly with `SurfacePlot`/`Layer` - useful for understanding the mechanics (hemisphere splitting, the dipole->dense inpainting, the lazy provider), but verbose for everyday work. `BrainPlot` packages it: two constructors (surfaces directly, or `from_source_space`), hemisphere selection, and - unlike independent `SurfacePlot`s - both hemispheres of *one* `BrainPlot` share a single color scale by default, since they're the same quantity, not two separate plots.

### From FreeSurfer surfaces directly - no source space involved

`BrainPlot`'s primary constructor takes surfaces as-is. `dm.load(subject, "white")` returns exactly the `(vertices, faces)` shape it expects per hemisphere - the same shape `mne.SourceSpaces` uses internally, which is why one conversion path (`surf_to_polydata`) serves both. Here the scalar is real per-vertex cortical surface area, not anything derived from a source space.

In [32]:
white_lh, white_rh = dm.load("intra12", "white")
area_lh, area_rh = dm.load("intra12", "area")

BrainPlot([white_lh, white_rh], [area_lh, area_rh], title="Cortical surface area", scalar_name="Area").show()


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad358c4fd50_31&reconnect=auto" class="pyv…

A single hemisphere works the same way - just pass one surface instead of two.

In [33]:
BrainPlot(white_lh, area_lh, title="Left hemisphere only").show()


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad366a11b50_32&reconnect=auto" class="pyv…

### From a source space - `BrainPlot.from_source_space`

Replaces the entire "both hemispheres, side by side" cell above (hand-building `hemi_r`/`map_r`/`provider_l`/`provider_r`, two `SurfacePlot`s, `|`, then `link_scale`) with one call. The dipole->dense mapping still happens lazily under the hood (see `viz/brain.py::_inpaint_dipole_values`) - this is a thinner way to write the same thing, not a different mechanism. Since values come from a callable provider, the color range still can't be auto-inferred, so we pass it explicitly via `mesh_params` - same reason as before, just one call now instead of a `link_scale` step afterward.

In [34]:
clim = (0, np.abs(stc.data).max())

brain = BrainPlot.from_source_space(
    src, stc.data, states=stc.times, hemi="both",
    scalar_name="Source Activity", mesh_params={"clim": clim, "cmap": "Reds"},
)
brain.show(slider_label="Time [s]")


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad353d1f950_33&reconnect=auto" class="pyv…

### Side-by-side comparison: the old `plot_brain_activity`

For directly comparing slider responsiveness against the pre-rewrite implementation (`mindmap/viz/OLD/`, kept for reference, no longer part of the public API). Same subject, same cropped 100-timepoint window, same hemispheres, same color scale - the only thing that differs is which code built the scene.

Importing it required a small fix first: `OLD/dispatch.py` and `OLD/vista.py` still imported `mindmap.viz.utils`/`mindmap.viz.vista`, which no longer exist at those paths since the rewrite moved them to `mindmap.viz.OLD.utils`/`mindmap.viz.OLD.vista` - two import lines corrected, no logic touched.

In [35]:
from mindmap.viz.OLD.dispatch import plot_brain_activity

plot_brain_activity(src, stc, hemi="both", mesh_params={"clim": clim, "cmap": "Reds"}).show()


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad36519f1d0_34&reconnect=auto" class="pyv…

`hemi="lh"` skips mapping the other hemisphere entirely - the original motivating case for this whole framework, now one keyword argument.

In [36]:
BrainPlot.from_source_space(src, stc.data, states=stc.times, hemi="lh",
                             scalar_name="Source Activity").show(slider_label="Time [s]")


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad344b249d0_35&reconnect=auto" class="pyv…

### `BrainPlot` + electrodes, the shorter way

`from_source_space` with no `values` is anatomy-only (just the shape) - combine it with a plain `SurfacePlot` for the electrodes, same `+` as before.

In [37]:
brain = BrainPlot.from_source_space(src, hemi="lh", mesh_params={"color": "white", "opacity": 0.3})
electrodes = SurfacePlot(pv.PolyData(positions), ave.data, states=ave.times, mesh_params={"cmap": "bkr"},
                          scalar_name="iEEG Potential")

(brain + electrodes).show(slider_label="Time [s]")


Widget(value='<iframe src="http://localhost:43059/index.html?ui=P_0x7ad365909190_36&reconnect=auto" class="pyv…

### A note on slider responsiveness in Jupyter, and `jupyter_backend`

If the slider above feels sluggish while dragging, it's very likely not this framework - it's PyVista's default Jupyter backend ("trame"), which streams the updated scene to your browser on *every* render, and `interaction_event="always"` (the default) re-renders on every pixel of drag movement. That cost scales with mesh size, and a full-resolution brain surface is large (~291k points across both hemispheres). Investigating this surfaced two independent, additive things worth knowing:

1. **PyVista version matters.** A side-by-side comparison of this exact cell against pyvista 0.46.3 vs. the version installed here showed a real difference - something in a later pyvista release made the trame path measurably slower, independent of anything in this framework (`plot_brain_activity`, the pre-rewrite function, showed the identical slowdown when run under the same version). Worth keeping in mind if you're chasing responsiveness and can control which pyvista version you install.

2. **`.show()` now takes `jupyter_backend`.** Pass `jupyter_backend="none"` to open a native OS render window instead of the inline browser widget - no streaming cost per frame, so dragging is as smooth as running outside Jupyter entirely. This needs a real display reachable from the kernel: on a genuinely remote/headless kernel it doesn't just show nothing, it **hangs** (PyVista waits on a window that will never appear) - so the cell below is commented out rather than run automatically. Uncomment and run it yourself when you have a display to test against.

In [38]:
brain = BrainPlot.from_source_space(src, stc.data, states=stc.times, hemi="both",
                                     scalar_name="Source Activity", mesh_params={"clim": clim, "cmap": "Reds"})

# Uncomment to try it yourself (needs a real display - see note above):
# brain.show(slider_label="Time [s]", jupyter_backend="none")


## 📄 - Fixed, publication-ready images: `view` and `.screenshot()`

Everything above is for exploring interactively. Papers need a fixed camera and a fixed timepoint, reproducible from a script - no manual "rotate until it looks right" step.

`BrainPlot(..., view=...)` sets an orthographic camera (no perspective distortion), auto-framed tightly to that panel's own content regardless of subject/mesh size - not a hardcoded number. `"lateral"`/`"medial"` are resolved relative to `hemi` (the medial surface of the right hemisphere faces the opposite direction from the left's - that flip is handled for you); absolute views (`"superior"`, `"anterior"`, ...) ignore `hemi` entirely. `.screenshot(path, state=...)` renders one specific state, off-screen, with no slider/key widgets cluttering the image.

In [39]:
BrainPlot.from_source_space(
    src, stc.data, states=stc.times, hemi="lh", view="lateral",
    scalar_name="Source Activity", mesh_params={"clim": clim, "cmap": "Reds"},
).screenshot(state=2.08, window_size=(800, 800))


array([[[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       ...,

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]]

### The standard 4-panel figure

LH lateral | LH medial | RH medial | RH lateral - medial surfaces face each other in the middle, the conventional layout. Each panel is its own `BrainPlot` with its own `hemi`/`view`; `|` lays them out, one `.screenshot()` call exports the whole row at a chosen timepoint.

In [40]:
lh_lateral = BrainPlot.from_source_space(src, stc.data, states=stc.times, hemi="lh", view="lateral",
                                          title="LH lateral", scalar_name="Source Activity", mesh_params={"clim": clim, "cmap": "Reds"})
lh_medial  = BrainPlot.from_source_space(src, stc.data, states=stc.times, hemi="lh", view="medial",
                                          title="LH medial", scalar_name="Source Activity", mesh_params={"clim": clim, "cmap": "Reds"})
rh_medial  = BrainPlot.from_source_space(src, stc.data, states=stc.times, hemi="rh", view="medial",
                                          title="RH medial", scalar_name="Source Activity", mesh_params={"clim": clim, "cmap": "Reds"})
rh_lateral = BrainPlot.from_source_space(src, stc.data, states=stc.times, hemi="rh", view="lateral",
                                          title="RH lateral", scalar_name="Source Activity", mesh_params={"clim": clim, "cmap": "Reds"})

figure = lh_lateral | lh_medial | rh_medial | rh_lateral
figure.screenshot("figure1.png", state=2.08, window_size=(2400, 700))


array([[[  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        ...,
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0]],

       [[  0,   0,   0],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [  0,   0,   0]],

       [[  0,   0,   0],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [  0,   0,   0]],

       ...,

       [[  0,   0,   0],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [  0,   0,   0]],

       [[  0,   0,   0],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [  0,   0,   0]],

       [[  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        ...,
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0]]

`window_size` matters here beyond just output resolution: the tight orthographic fit is computed from each subplot's actual pixel aspect ratio, so it has to be set *before* fitting - `.screenshot(window_size=...)` handles that ordering correctly (passing `window_size` straight to a raw `pv.Plotter.screenshot()` call instead would compute the fit against the wrong, still-default window size and leave every panel looking too small - a real bug this framework hit and fixed while building this).

`&` composes with `.screenshot()` too - a dorsal/ventral row underneath would just be `figure & (dorsal_row)`, once you have panels for those views.